In [1]:
import gym
import gym_mppt
from gym import envs
import numpy as np
import stable_baselines3
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

import os
models_dir = "models/DDPG"
logdir = "logs"

if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    
if not os.path.exists(logdir):
    os.makedirs(logdir)  

In [2]:
env = gym.make("mppt_shaded-v2")

num_states = len(env.observation_space)
print("Size of State Space ->  {}".format(num_states))
print(env.observation_space)

print()

num_actions = env.action_space.shape[0]
print("Size of Action Space ->  {}".format(num_actions))
print(env.action_space)

upper_bound = env.action_space.high[0]
lower_bound = env.action_space.low[0]

print("Max Value of Action ->  {}".format(upper_bound))
print("Min Value of Action ->  {}".format(lower_bound))

Size of State Space ->  3
Dict(Power:Box([0.], [inf], (1,), float32), Voltage:Box([0.], [212.], (1,), float32), deltaPower:Box([-inf], [inf], (1,), float32))

Size of Action Space ->  1
Box([-10.], [10.], (1,), float32)
Max Value of Action ->  10.0
Min Value of Action ->  -10.0


In [3]:
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

tensorboard_log = os.path.join(logdir, "tensorboard")
model = DDPG("MultiInputPolicy", env, action_noise=action_noise, verbose=1, batch_size=256, learning_rate=10, tensorboard_log=tensorboard_log)
# TIMESTEPS = 400001
TIMESTEPS = 1000001
model.learn(total_timesteps=TIMESTEPS, log_interval=10, progress_bar=True, reset_num_timesteps=False, tb_log_name="DDPG-successor5")
model.save(f"{models_dir}/{TIMESTEPS}")
env = model.get_env()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to logs\tensorboard\DDPG-successor5_0


Output()

c:\users\archi\python code\btp\pvcellrltorch_v2\mppt-gym_v2\gym_mppt\envs\mppt_env_shaded2.py:37: RuntimeWarning: 
overflow encountered in exp
  I = self.Np * Iph - self.Np * Irs * (np.exp(self.q / (self.k * Tcell * self.A) * vx * (1/SH) / self.Ns) - 1)

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -83.2     |
| time/              |           |
|    episodes        | 10        |
|    fps             | 108       |
|    time_elapsed    | 18        |
|    total_timesteps | 2000      |
| train/             |           |
|    actor_loss      | -4.02e+08 |
|    critic_loss     | 2.37e+16  |
|    learning_rate   | 10        |
|    n_updates       | 1800      |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -84.8     |
| time/              |           |
|    episodes        | 20        |
|    fps             | 105       |
|    time_elapsed    | 37        |
|    total_timesteps | 4000      |
| train/             |           |
|    actor_loss      | -3.75e+08 |
|    critic_loss     | 4.47e+15  |
|    learning_rate   | 10        |
|    n_updates      

KeyboardInterrupt: 

In [ ]:
for i in range(10):
    obs = env.reset()
    for j in range(100):
        action, _states = model.predict(obs)
        obs, rewards, dones, info = env.step(action)
        print(obs)
    print()